In [ ]:
# 
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [ ]:
passmark = 40

In [ ]:
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")
factor = 1000
df = pd.concat([df]*factor)
df.info()

In [ ]:
### cell 1 ###

df.isna().sum()

In [ ]:
### cell 2 ###

df.head()

In [ ]:
### cell 3 ###

df.describe()

In [ ]:
### cell 4 ###

df.isnull().sum()

In [ ]:
### cell 5 ###

df['math score'] = df['math score'].astype(float).fillna(-1)  # Replace NaNs with a default value

# Vectorized conditional assignment using cuDF
df['Math_PassStatus'] = (df['math score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's optimized value_counts()
df['Math_PassStatus'].value_counts()

In [ ]:
### cell 6 ###

#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()

In [ ]:
### cell 7 ###

#rewritten
# Convert 'reading score' to numeric (cuDF does not have errors='coerce', so use fillna after)
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed

# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's value_counts() (faster than pandas on GPU)
df['Reading_PassStatus'].value_counts()

##rewritten
df['writing score'] = df['writing score'].astype('float32')

# Use cuDF's boolean indexing + direct assignment (avoids unnecessary operations)
df['Writing_PassStatus'] = 'F'  # Default all to 'F' first
df.loc[df['writing score'] >= passmark, 'Writing_PassStatus'] = 'P'  # Assign 'P' where needed

# Efficient value_counts() with cuDF
df['Writing_PassStatus'].value_counts()

In [ ]:
### cell 8 ###

df['OverAll_PassStatus'] = ((df['Math_PassStatus'] == 'P') & 
                            (df['Reading_PassStatus'] == 'P') & 
                            (df['Writing_PassStatus'] == 'P')).astype("str").replace({'True': 'P', 'False': 'F'})

# Optimized cuDF value_counts()
df['OverAll_PassStatus'].value_counts()

In [ ]:
### cell 9 ###

df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

In [ ]:
### cell 10 ###

##rewritten
# Initialize Grade column with 'F' (default for failures)
df['Grade'] = 'F'

# Apply vectorized conditional assignments
df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'
df.loc[(df['Percentage'] >= 80) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'A'
df.loc[(df['Percentage'] >= 70) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'B'
df.loc[(df['Percentage'] >= 60) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'C'
df.loc[(df['Percentage'] >= 50) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'D'
df.loc[(df['Percentage'] >= 40) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'E'

# Efficient value counts
df['Grade'].value_counts()